# 01 — H100 Video Profiling and Temporal Lane Demo

Loads a saved checkpoint, runs video inference, performs lane association across frames, applies temporal smoothing, saves an overlay video, and exports per-second profiling stats.

In [ ]:
!pip install -q torch torchvision pynvml opencv-python matplotlib pyyaml scipy

In [ ]:

import os, sys, time, csv, cv2, math, shutil, threading
import numpy as np
import torch
from google.colab import drive
drive.mount('/content/drive')

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
VIDEO_DIR = os.path.join(ECOCAR_ROOT, "video")
WEIGHTS_DIR = os.path.join(ECOCAR_ROOT, "weights")
OUTPUT_DIR = os.path.join(ECOCAR_ROOT, "profiling")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROJECT_DIR = os.path.join(ECOCAR_ROOT, "DETR_GeoLane_pipeline")
if not os.path.isdir(PROJECT_DIR):
    !git clone https://github.com/ChenSiyun1234/EcoCAR-Perception-Pipeline-YOLO26-BDD100K.git /content/repo 2>/dev/null || true
    PROJECT_DIR = "/content/repo/DETR_GeoLane_pipeline"
sys.path.insert(0, PROJECT_DIR)

print('Project dir:', PROJECT_DIR)
print('Video dir:', VIDEO_DIR)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")


In [ ]:

import pynvml
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

class GPUMonitor:
    def __init__(self, interval_ms=100):
        self.interval = interval_ms / 1000.0
        self.samples = []
        self._running = False
    def start(self):
        self._running = True
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
    def stop(self):
        self._running = False
        if hasattr(self, '_thread'):
            self._thread.join(timeout=2.0)
    def _poll(self):
        while self._running:
            try:
                util = pynvml.nvmlDeviceGetUtilizationRates(handle)
                mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
                self.samples.append({'timestamp': time.time(), 'gpu_util': util.gpu, 'mem_used_mb': mem.used / (1024**2)})
            except Exception:
                pass
            time.sleep(self.interval)
    def per_second_report(self, t0):
        buckets = {}
        for s in self.samples:
            sec = int(s['timestamp'] - t0)
            buckets.setdefault(sec, []).append(s)
        report = []
        for sec in sorted(buckets):
            vals = buckets[sec]
            report.append({
                'second': sec,
                'gpu_util_avg': float(np.mean([v['gpu_util'] for v in vals])),
                'gpu_util_max': float(np.max([v['gpu_util'] for v in vals])),
                'mem_used_avg_mb': float(np.mean([v['mem_used_mb'] for v in vals])),
            })
        return report


In [ ]:

from src.config import Config
from src.model import build_model
from src.visualize import draw_detections
from src.temporal_utils import associate_lanes, smooth_lane_points

# Find checkpoint
ckpt_candidates = [
    os.path.join(WEIGHTS_DIR, 'dualpath_v1_best.pt'),
    os.path.join(WEIGHTS_DIR, 'dualpath_v1_last.pt'),
]
ckpt_path = next((p for p in ckpt_candidates if os.path.isfile(p)), None)
assert ckpt_path is not None, f'No checkpoint found in {WEIGHTS_DIR}'
ckpt = torch.load(ckpt_path, map_location='cuda' if torch.cuda.is_available() else 'cpu', weights_only=False)
saved_cfg = ckpt.get('config', {})
cfg = Config.from_dict(saved_cfg) if saved_cfg else Config()
model = build_model(cfg)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()
print('Loaded checkpoint:', ckpt_path)
print('Epoch:', ckpt.get('epoch'))


In [ ]:

# Find input video
video_files = sorted([os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR)]) if os.path.isdir(VIDEO_DIR) else []
video_files = [f for f in video_files if f.lower().endswith(('.mp4','.avi','.mov','.mkv'))]
assert video_files, f'No videos found in {VIDEO_DIR}'
INPUT_VIDEO = video_files[0]
cap = cv2.VideoCapture(INPUT_VIDEO)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_vid = cap.get(cv2.CAP_PROP_FPS) or 30.0
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print('Video:', os.path.basename(INPUT_VIDEO), total_frames, 'frames @', fps_vid, 'FPS')

out_video = os.path.join(OUTPUT_DIR, os.path.splitext(os.path.basename(INPUT_VIDEO))[0] + '_overlay.mp4')
writer = cv2.VideoWriter(out_video, cv2.VideoWriter_fourcc(*'mp4v'), fps_vid, (src_w, src_h))


In [ ]:

def decode_lanes(outputs, img_w, img_h, exist_thresh=0.5):
    pred_points = outputs['lane_pred_points'][0].detach().cpu().numpy()
    exist = outputs['lane_exist_logits'][0,:,0].sigmoid().detach().cpu().numpy()
    types = outputs['lane_type_logits'][0].argmax(dim=-1).detach().cpu().numpy()
    lanes = []
    for qi, conf in enumerate(exist):
        if conf < exist_thresh:
            continue
        pts = pred_points[qi].copy()
        pts[:,0] *= img_w; pts[:,1] *= img_h
        lanes.append({'points': pts, 'conf': float(conf), 'type': int(types[qi]), 'id': None})
    return lanes

def draw_lanes_with_ids(frame_bgr, lanes):
    canvas = frame_bgr.copy()
    for lane in lanes:
        pts = lane['points'].astype(np.int32)
        color = (0,255,255) if lane['id'] is None else ((37 * lane['id']) % 255, (97 * lane['id']) % 255, (167 * lane['id']) % 255)
        if len(pts) >= 2:
            cv2.polylines(canvas, [pts], False, color, 3)
            x, y = pts[min(len(pts)-1, 5)]
            cv2.putText(canvas, f"lane {lane['id']}", (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return canvas


In [ ]:

# Warmup
dummy = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=device)
for _ in range(5):
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(device == 'cuda')):
        _ = model(dummy)
if device == 'cuda':
    torch.cuda.synchronize()
print('Warmup done.')

monitor = GPUMonitor(interval_ms=100)
frame_times = []
temporal_rows = []
prev_lanes = []
next_lane_id = 0
t_start = time.time()
monitor.start()
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    t0 = time.time()
    rgb = cv2.cvtColor(cv2.resize(frame, (cfg.img_size, cfg.img_size)), cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(device == 'cuda')):
        outputs = model(tensor)
    if device == 'cuda':
        torch.cuda.synchronize()
    frame_times.append((time.time() - t0) * 1000.0)

    # detections on resized frame then upscale
    det_vis = draw_detections(rgb, outputs['det_pred_logits'][0], outputs['det_pred_boxes'][0], conf_thresh=cfg.conf_thresh, use_expanded=cfg.use_expanded_classes)
    det_vis = cv2.cvtColor(det_vis, cv2.COLOR_RGB2BGR)
    det_vis = cv2.resize(det_vis, (src_w, src_h))

    curr_lanes = decode_lanes(outputs, src_w, src_h, exist_thresh=0.5)
    matches, unmatched_curr = associate_lanes(prev_lanes, curr_lanes, dist_thresh_px=40.0)
    matched_curr = set()
    for pi, cj, dist in matches:
        curr_lanes[cj]['id'] = prev_lanes[pi]['id']
        curr_lanes[cj]['points'] = smooth_lane_points(prev_lanes[pi]['points'], curr_lanes[cj]['points'], alpha=0.65)
        matched_curr.add(cj)
        temporal_rows.append({'frame': frame_idx, 'lane_id': curr_lanes[cj]['id'], 'assoc_dist_px': dist, 'status': 'matched'})
    for cj in unmatched_curr:
        curr_lanes[cj]['id'] = next_lane_id
        temporal_rows.append({'frame': frame_idx, 'lane_id': next_lane_id, 'assoc_dist_px': '', 'status': 'new'})
        next_lane_id += 1
    overlay = draw_lanes_with_ids(det_vis, curr_lanes)
    writer.write(overlay)
    prev_lanes = [{'id': lane['id'], 'points': lane['points'].copy()} for lane in curr_lanes]
    frame_idx += 1
    if frame_idx % 100 == 0:
        print(f'Processed {frame_idx}/{total_frames}')

if device == 'cuda':
    torch.cuda.synchronize()
t_end = time.time()
monitor.stop()
cap.release(); writer.release()
total_time = max(t_end - t_start, 1e-6)
avg_fps = frame_idx / total_time
avg_lat = float(np.mean(frame_times)) if frame_times else 0.0
p95_lat = float(np.percentile(frame_times, 95)) if frame_times else 0.0
p99_lat = float(np.percentile(frame_times, 99)) if frame_times else 0.0
print('Video written to:', out_video)


In [ ]:

# Save reports
report = monitor.per_second_report(t_start)
gpu_csv = os.path.join(OUTPUT_DIR, 'gpu_per_second.csv')
with open(gpu_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['second','gpu_util_avg','gpu_util_max','mem_used_avg_mb'])
    w.writeheader(); w.writerows(report)
lat_csv = os.path.join(OUTPUT_DIR, 'frame_latencies.csv')
with open(lat_csv, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['frame','latency_ms'])
    for i, t in enumerate(frame_times):
        w.writerow([i, f'{t:.3f}'])
assoc_csv = os.path.join(OUTPUT_DIR, 'lane_temporal_association.csv')
with open(assoc_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['frame','lane_id','assoc_dist_px','status'])
    w.writeheader(); w.writerows(temporal_rows)
print(gpu_csv); print(lat_csv); print(assoc_csv)


In [ ]:

        import matplotlib.pyplot as plt
        seconds = [r['second'] for r in report]
        gpu_avg = [r['gpu_util_avg'] for r in report]
        gpu_max = [r['gpu_util_max'] for r in report]
        mem_gb = [r['mem_used_avg_mb']/1024.0 for r in report]
        fig, axes = plt.subplots(2,2, figsize=(16,10))
        axes[0,0].plot(seconds, gpu_avg, label='avg'); axes[0,0].plot(seconds, gpu_max, label='max'); axes[0,0].legend(); axes[0,0].set_title('GPU util / sec')
        axes[0,1].plot(seconds, mem_gb); axes[0,1].set_title('GPU mem GB / sec')
        axes[1,0].hist(frame_times, bins=40); axes[1,0].set_title('Frame latency ms')
        roll = [1000.0 / np.mean(frame_times[max(0, i-29):i+1]) for i in range(len(frame_times))] if frame_times else []
        axes[1,1].plot(roll); axes[1,1].set_title('Rolling FPS')
        plt.tight_layout()
        fig_path = os.path.join(OUTPUT_DIR, 'gpu_profile.png')
        plt.savefig(fig_path, dpi=160)
        plt.show()
        print('
' + '='*55)
        print('GPU PROFILING SUMMARY')
        print('='*55)
        print('Frames:', frame_idx)
        print('Avg FPS:', round(avg_fps, 2))
        print('Avg latency:', round(avg_lat, 2), 'ms')
        print('P95 latency:', round(p95_lat, 2), 'ms')
        print('P99 latency:', round(p99_lat, 2), 'ms')
        print('Overlay video:', out_video)
        print('Profiling dir:', OUTPUT_DIR)
        print('='*55)
